In [17]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np


mnist = fetch_openml('mnist_784', version=1, as_frame=False) # X: (70000, 784)
X, y = mnist["data"], mnist["target"]
X = X.astype(np.float32) / 255.0
y = y.astype(np.int64)
# scale to [0, 1]
X_trainval, X_test, y_trainval, y_test = train_test_split(
X, y, test_size=10000, random_state=42, stratify=y
)


# MLP Model

In [18]:
import os
import torch
from torch import nn

class MLP_Model(nn.Module):
    def __init__(self):
        super(MLP_Model, self).__init__()
        self.input = nn.Linear(784,2 * 784 )
        self.hidden = nn.Linear(2*784, 2*784)
        self.output = nn.Linear(2*784, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.input(x))
        x = self.relu(self.hidden(x))
        x = self.output(x)

        return x


os.makedirs('Models/lr', exist_ok=True)

# optuna for Learning Rate

In [19]:
import optuna
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.model_selection import KFold

# --- Data ---
X_trainval_tensor = torch.from_numpy(X_trainval).float()
y_trainval_tensor = torch.from_numpy(y_trainval).long()
dataset = TensorDataset(X_trainval_tensor, y_trainval_tensor)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SPLITS = 5
N_EPOCHS = 10
BATCH    = 1024


def _train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        criterion(model(X), y).backward()
        optimizer.step()

def _calculate_val_loss(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for X,y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            output = model(X)
            loss = criterion(output, y)
            running_loss += loss.item()
    avg_loss = running_loss / len(loader)

    return avg_loss

def objetive_lr(trial:optuna.Trial) -> float:
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-1, log=True)

    criterion = nn.CrossEntropyLoss()
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    fold_loss = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_trainval_tensor)):
        train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH, shuffle=True)
        val_loader = DataLoader(Subset(dataset, val_idx), batch_size=len(val_idx), shuffle=False)

        model = MLP_Model().to(DEVICE)
        optimizer = optim.SGD(model.parameters(), lr=lr)

        for epoch in range(N_EPOCHS):
            _train_one_epoch(model=model, loader=train_loader, optimizer=optimizer, criterion=criterion)

        fold_loss.append(_calculate_val_loss(model=model, loader=val_loader, criterion=criterion))

    return float(np.mean(fold_loss))

study = optuna.create_study(direction="minimize")
study.optimize(objetive_lr, n_trials=25, show_progress_bar=True)

print("Best parameters:", study.best_params)
print("Mean avg val loss CV: ", study.best_value)

[I 2026-03-29 20:17:45,876] A new study created in memory with name: no-name-25976188-8861-4451-b695-e32fa371d4cf


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-03-29 20:18:22,391] Trial 0 finished with value: 0.27501052618026733 and parameters: {'learning_rate': 0.09763027894152929}. Best is trial 0 with value: 0.27501052618026733.
[I 2026-03-29 20:18:56,191] Trial 1 finished with value: 2.283381986618042 and parameters: {'learning_rate': 0.0004193298907099138}. Best is trial 1 with value: 2.283381986618042.
[I 2026-03-29 20:19:35,112] Trial 2 finished with value: 0.23151707351207734 and parameters: {'learning_rate': 0.14566198251597595}. Best is trial 1 with value: 2.283381986618042.
[I 2026-03-29 20:20:12,508] Trial 3 finished with value: 0.24477218687534333 and parameters: {'learning_rate': 0.1319381659469213}. Best is trial 1 with value: 2.283381986618042.
[I 2026-03-29 20:20:52,144] Trial 4 finished with value: 2.2737475872039794 and parameters: {'learning_rate': 0.0006517532048972725}. Best is trial 1 with value: 2.283381986618042.
[I 2026-03-29 20:21:32,966] Trial 5 finished with value: 2.21612663269043 and parameters: {'learni

# Training Model with new lr

In [22]:
from matplotlib import pyplot as plt

N_EPOCHS = 100

folds = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, val_index) in enumerate(folds.split(X_trainval)):
    print(f"---FOLD {fold} ---")

    # Fold Split and Loaders
    train_sample = Subset(dataset, train_index)
    val_sample = Subset(dataset, val_index)

    train_loader = DataLoader(train_sample, batch_size=BATCH, shuffle=True)
    val_loader = DataLoader(val_sample, batch_size=BATCH, shuffle=False)

    #Model, Optimizer and Loss
    model = MLP_Model().to(DEVICE)
    optimizer = optim.SGD(model.parameters(), lr=study.best_params['learning_rate'])
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    history = {'train_loss':[], 'val_loss':[]}

    # Training Loop
    for epoch in range(N_EPOCHS):

        model.train()
        train_running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            train_running_loss += loss.item()

        avg_train_loss = train_running_loss / len(train_loader)

        # Model Validation
        model.eval()
        val_running_loss = 0.0
        with torch.no_grad():
            for x,y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                output = model(x)
                loss = criterion(output, y)
                val_running_loss += loss.item()

        avg_val_loss = val_running_loss / len(val_loader)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)

        # Saving Best Model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            checkpoint = {
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'history': history
            }
            PATH = os.path.join('Models', 'lr', f'model2_{fold}.pth')
            torch.save(checkpoint, PATH)
            #print(f"Epoch {epoch}: Model Saved with new best loss {avg_val_loss:.4f}")


    # Vizualization
    plt.figure(figsize=(8, 5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title(f'Loss Curve - Fold {fold}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

---FOLD 0 ---


KeyboardInterrupt: 